# TRM Compiler Pass Ordering — Real LLVM Training

Train a 60K-param TRM model on **real LLVM** via CompilerGym, benchmark vs LLVM -Oz/-O3.

In [ ]:
# Cell 1: Clone repo
import os
import subprocess

PROJECT_DIR = '/content/trm-youtubevids'
VENV_DIR = '/content/trm-env'

if not os.path.exists(PROJECT_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/Cree0618/trm-youtubevids.git', PROJECT_DIR], check=True)
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull'], check=True)

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Cell 2: Create Python 3.11 venv and install dependencies
# compiler_gym requires Python <= 3.11 (uses distutils removed in 3.12)

import subprocess
import os

# Create venv if needed
if not os.path.exists(VENV_DIR):
    result = subprocess.run(['python3.11', '-m', 'venv', VENV_DIR], capture_output=True, text=True)
    print(f"Created venv: {result.returncode == 0}")
else:
    print(f"Venv exists: {VENV_DIR}")

VENV_PIP = f'{VENV_DIR}/bin/pip'
VENV_PYTHON = f'{VENV_DIR}/bin/python'

# Upgrade pip
subprocess.run([VENV_PIP, 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)

# Install torch (CPU version to avoid CUDA issues)
subprocess.run([VENV_PIP, 'install', 'torch', 'numpy', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)

# Install grpcio NEWER version (has pre-built wheels for Python 3.11)
# The trick: install newer grpcio FIRST, then install compiler_gym with --no-deps
subprocess.run([VENV_PIP, 'install', 'grpcio'], check=True)

# Install compiler_gym without dependencies (grpcio version conflict)
subprocess.run([VENV_PIP, 'install', 'compiler_gym', '--no-deps'], check=True)

# Verify
result = subprocess.run([VENV_PYTHON, '-c', 'import compiler_gym; print(f"CompilerGym: {compiler_gym.__version__}")'], capture_output=True, text=True)
print(result.stdout.strip())
if result.returncode != 0:
    print(f"Error: {result.stderr}")

In [ ]:
# Cell 3: Quick smoke test - verify CompilerGym works

import subprocess

VENV_PYTHON = '/content/trm-env/bin/python'

test_code = '''
import compiler_gym
env = compiler_gym.make("llvm-v0", benchmark="cbench-v1/qsort",
    observation_space="Autophase", reward_space="IrInstructionCountOz")
obs = env.reset()
ap = env.observation["Autophase"]
ic = env.observation["IrInstructionCount"]
print(f"Autophase: {len(ap)} features, Initial inst: {ic}")
for i in range(3):
    _, reward, done, _ = env.step(env.action_space.sample())
    inst = env.observation["IrInstructionCount"]
    print(f"  Step {i}: inst={inst} reward={reward:.2f} done={done}")
env.close()
print("CompilerGym OK!")
'''

result = subprocess.run([VENV_PYTHON, '-c', test_code], capture_output=False, text=True)

In [ ]:
# Cell 4: Run TRM on REAL LLVM - train and benchmark

import subprocess

VENV_PYTHON = '/content/trm-env/bin/python'
PROJECT_DIR = '/content/trm-youtubevids'

cmd = [
    VENV_PYTHON,
    f'{PROJECT_DIR}/trm_compiler_real_llvm.py',
    '--epochs', '10',
    '--episodes', '10',
    '--benchmarks', 'qsort', 'adpcm',
    '--max-steps', '20',
    '--batch-size', '64',
    '--num-random', '20',
    '--seed', '42'
]

# Note: No --synthetic flag = uses real CompilerGym
result = subprocess.run(cmd, capture_output=False, text=True)
print(f"\nDone. Exit code: {result}")

## Results

The script outputs benchmark tables comparing:
- **LLVM-Oz**: Fixed -Oz pipeline
- **LLVM-O3**: Fixed -O3 pipeline
- **Random(N)**: Best of N random trials
- **TRM-loop**: TRM with closed-loop feedback
- **TRM-blind**: TRM without feedback